# 01 — Data Understanding

**Tujuan:** memahami struktur dan kualitas dataset `crm_50000_customers_dirty_v3.csv` **sebelum** melakukan standardization atau matching.

**Aturan yang berlaku di notebook ini:**
- Tidak ada asumsi nama kolom/jumlah baris/duplicate rate sebelum dihitung langsung.
- Raw data tidak diubah sama sekali di sini (read-only).
- Belum ada blocking, Splink matching, threshold, atau entity clustering di notebook ini.
- Semua angka di bawah adalah fakta dataset (dihitung langsung), bukan klaim dari deskripsi Kaggle.

In [ ]:
from gettext import install
import pip
pip.main(['install', 'numpy', 'pandas'])


Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.


Collecting numpy

Using cached numpy-2.5.3-cp314-cp314-win_amd64.whl.metadata (6.6 kB)

Collecting pandas

Downloading pandas-3.0.6-cp314-cp314-win_amd64.whl.metadata (19 kB)

...

c:\Users\User\Downloads\Fix\.venv\Lib\site-packages\pip\_vendor\rich\live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

DEPRECATION: Unexpected import of 'ipywidgets' after pip install started. pip 26.3 will enforce this behaviour change. Discussion can be found at https://github.com/pypa/pip/issues/13842 (c:\Users\User\Downloads\Fix\.venv\Lib\site-packages\pip\_vendor\rich\live.py:252)


c:\Users\User\Downloads\Fix\.venv\Lib\site-packages\pip\_vendor\rich\live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Successfully installed numpy-2.5.3 pandas-3.0.6 tzdata-2026.4

0

In [8]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

RAW_PATH = r"C:\Users\User\Downloads\Fix\data\raw\crm_50000_customers_dirty_v3.csv"


## A. Load dataset

In [4]:
try:
    df_raw = pd.read_csv(RAW_PATH, sep=";", encoding="utf-8")
    print(f"Berhasil load: {RAW_PATH}")
except Exception as e:
    print(f"GAGAL load dataset dari {RAW_PATH}")
    raise e

print(f"Jumlah rows    : {df_raw.shape[0]:,}")
print(f"Jumlah columns : {df_raw.shape[1]}")


Berhasil load: C:\Users\User\Downloads\Fix\data\raw\crm_50000_customers_dirty_v3.csv
Jumlah rows    : 50,000
Jumlah columns : 14


## B. Schema inspection

In [6]:
print("Nama kolom aktual:")
for i, col in enumerate(df_raw.columns):
    print(f"  [{i}] {col}  -> dtype: {df_raw[col].dtype}")


Nama kolom aktual:
  [0] customer_id  -> dtype: str
  [1] first_name  -> dtype: str
  [2] last_name  -> dtype: str
  [3] email  -> dtype: str
  [4] phone_number  -> dtype: str
  [5] gender  -> dtype: str
  [6] dob  -> dtype: str
  [7] signup_date  -> dtype: str
  [8] address  -> dtype: str
  [9] city  -> dtype: str
  [10] state  -> dtype: str
  [11] country  -> dtype: str
  [12] device_id(s)  -> dtype: str
  [13] source  -> dtype: str


In [9]:
df_raw.head(10)


,customer_id,first_name,last_name,email,phone_number,gender,dob,signup_date,address,city,state,country,device_id(s),source
0,0d244537-8164-4b84-bcf2-0e43766f3221,mARIA,day,maria.day697@hotmail.com,449.977.1729x282,M,11/04/1988,05/12/2016,107 Joseph Station,Padillaborough,Oregon,Micronesia,56f3a452-b3f5-488b-b90d-8cb2b3343c70,referral
1,66c5800b-ad5c-4fac-86df-16472d82f02d,KKeevvin,Cantu,kevin.cantu968@gmail.com,501.347.4824x64633,M,19/09/1955,03/02/2024,3577 Anita Knoll Apt. 107,Justinton,Delaware,Portugal,cd83e240-0d50-4e67-b146-23deb6c49940,referral
2,e7852eee-0a28-46dc-aa34-445a02530746,Ronald,Daniel,ronald.daniel138@yahoo.com,+1-268-822-6332x26814,M,02/11/1971,12/04/2016,3440 Tammy Views Suite 698,New Autumnland,Mississippi,Uganda,369b8950-dfae-44ce-bbb4-d292f118be67,web
3,14f0e67d-e60e-4989-88cf-5be50d515a69,Katherine,Smith,katherine.smith357@yahoo.com,864-894-1347,F,25/03/1955,21/06/2019,98888 Morales Lakes,New Jose,Minnesota,Bahrain,3e83071f-53ed-4a05-9dfd-5ff2031443b5,web
4,1ce2f061-345c-4121-a4ba-32963cf9a15d,Brandon,Glover,brandon.glover804@hotmail.com,4.897.325.436,M,25/06/1968,09/01/2022,55037 Denise Crest Apt. 565,Port Amber,Oregon,Latvia,751d4410-3d73-4f4c-b432-10a0213b925f,referral
5,6ddc1772-f210-4111-87c9-c2ec73776c37,Toni,Li,toni.li092@gmail.com,(522)382-8559,F,30/09/1981,25/11/2023,1230 Padilla Parks Suite 647,North Melvintown,Kentucky,Uganda,f8013534-0730-4d00-8120-445e02ebed1f,app
6,9a3ae2d2-97d7-43b4-843c-ebac2fc0131c,Barbara,Luna,barbara.luna614@yahoo.com,001-063-045-0948x4636,F,07/01/1991,01/11/2025,768 Martha Plaza,Andersonberg,Maine,Guyana,f7bf7b38-f88f-4e85-b065-a163eb909a13,referral
7,69814e99-477d-48dc-8d7b-2d1137f1cf37,Richard,Benson,richard.benson220@yahoo.com,7.552.810.372,M,02/08/2002,27/02/2016,8812 Cheryl Radial Apt. 428,Huntview,Wyoming,Greenland,ceaa4bc8-5dc0-42d8-830f-5461e6fabc1e,web
8,d081d287-527e-4a77-bd62-50e28a110ddb,James,Ali,james.ali627@yahoo.com,1974876994,M,11/02/1968,08/04/2018,674 Lynch Spring Apt. 341,Martinstad,New Hampshire,Netherlands,6f5f7a22-9b27-4d01-b14e-ad09156419ab,app
9,5a298d80-e951-4565-98cb-5cb7ef301fc2,JAY,JJoonnes,jay.jones624@yahoo.com,-8823,M,22/08/1963,05/03/2023,82891 Grant Park Suite 174,Brownfort,Wisconsin,Liechtenstein,2d543635-2c5d-46c6-8c8e-5177b328ac31,web


In [10]:
# Sample acak
df_raw.sample(10, random_state=42)


,customer_id,first_name,last_name,email,phone_number,gender,dob,signup_date,address,city,state,country,device_id(s),source
33553,14b19db2-25f2-4c42-afac-822b0412f55c,Nicole,Potts,nicole.potts659@gmail.com,001-640-729-1553x3879,F,21/10/1971,10/05/2015,245 Myers Union Apt. 852,Schmidtview,New Jersey,Bosnia and Herzegovina,3fbc6171-6d49-4080-a482-c4c99af7e158,web
9427,11eae6a0-c04d-4e90-aaeb-c312cdd85f4c,Jessica*,HHaanncock,jessica.hancock630@gmail.com,2.306.799.769,F,16/12/1989,01/12/2020,686 Julia Union Apt. 529,West Brian,Oregon,Maldives,64bd5dca-1628-464d-a433-1a27dc42bfaa,app
199,c534fb8b-d99a-444c-943b-1ec88d67be00,Kimberly,Cox!,kimberly.cox676@yahoo.com,087-265-6604,F,04/07/2005,22/05/2015,442 Katrina Drives,North Andrea,Minnesota,Svalbard & Jan Mayen Islands,bc7bf030-33da-4093-93d8-404ab675f0ef,web
12447,bf14ab2c-de04-4141-894d-f9538c71a8f6,Danielle,Johnson,danielle.johnson048@yahoo.com,5.530.801.979,M,13/04/1971,12/08/2019,59419 Williams Squares Apt. 528,New Brandonton,Oklahoma,Belgium,24c21a12-7e3a-4a78-a143-e293eb669179,app
39489,96ba3e93-5bd6-44d0-8413-96af38b371a0,Darren,Rogers,darren.rogers723@yahoo.com,(264)226-4218,F,15/03/1998,31/05/2020,5919 Coleman Crest,Glendaland,Maryland,Australia,63a197f7-0092-40e2-a251-d75b81e4efcd,app
42724,98b84fac-290f-4e0d-bed0-7877e31236ff,Diane,Wilkins,diane.wilkins421@yahoo.com,001-272-370-9902x682,F,21/02/1966,06/03/2024,052 Patel Islands,New Brad,South Dakota,Jordan,6802f7bd-958e-4ab2-ad12-e350230ec1e2,web
10822,153e1fef-c8db-43cc-b92c-051cc6e7c299,Valerie,Walker,valerie.walker476@yahoo.com,+1-897-770-1091x7231,M,26/09/1999,17/04/2015,5137 Chapman Falls Apt. 415,Angelachester,Oklahoma,Japan,a1b27a41-d546-46ac-bc6f-25202acde3ea,referral
49498,a8ae12da-f250-4677-9305-09986b4bc8a6,Carla,Acevedo,carla.acevedo938@hotmail.com,+1-370-645-8796x99924,O,27/12/1996,20/06/2017,801 Jones Lights,New Timothybury,New Jersey,Djibouti,4ee56951-9555-4da6-9083-8f4ab527c143,app
4144,2123f54a-9d6e-43f9-a29b-1e7e6219fa05,Dylan,Manning,dylan.manning784@yahoo.com,+1-413-997-4858x0456,M,06/05/1999,01/04/2019,77227 Eric Isle Apt. 901,New Terristad,Alaska,Israel,6b87c436-ae2e-4958-ae3d-6f8ff7b3a2dd,web
36958,dbd0e933-8627-42c8-a0e3-028813cc2447,LINDA,ANDRADE,linda.andrade601@gmail.com,448-698-2727x632,F,24/01/1972,22/08/2018,7395 Miller Neck,Port Janiceshire,North Dakota,Kiribati,8cd42ba2-adf1-4e88-8306-c25dc44784c4,web


## C. Missing values

Jumlah dan persentase missing per kolom (dihitung langsung, bukan asumsi).

In [11]:
missing_summary = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_pct": (df_raw.isna().mean() * 100).round(2)
}).sort_values("missing_pct", ascending=False)

missing_summary


,missing_count,missing_pct
source,1251,2.50
email,1040,2.08
first_name,0,0.00
last_name,0,0.00
phone_number,0,0.00
customer_id,0,0.00
gender,0,0.00
dob,0,0.00
address,0,0.00
signup_date,0,0.00


In [12]:
# Cek juga string kosong / whitespace-only yang mungkin tidak terdeteksi sebagai NaN oleh pandas
def blank_like_count(series):
    if series.dtype != object:
        return 0
    return series.astype(str).str.strip().eq("").sum()

blank_summary = pd.Series({col: blank_like_count(df_raw[col]) for col in df_raw.columns}, name="blank_or_whitespace_only")
blank_summary[blank_summary > 0].sort_values(ascending=False)


Series([], Name: blank_or_whitespace_only, dtype: int64)

## D. Duplicate inspection

1. Exact duplicate rows
2. Duplicate berdasarkan field identity yang tersedia (dicek dinamis sesuai kolom aktual)
3. Uniqueness tiap field

**Catatan:** duplicate berdasarkan satu field (mis. email sama) TIDAK otomatis berarti duplicate customer — ini baru indikasi awal, bukan kesimpulan.

In [13]:
exact_dup_count = df_raw.duplicated(keep=False).sum()
print(f"Exact duplicate rows (seluruh kolom identik): {exact_dup_count:,}")


Exact duplicate rows (seluruh kolom identik): 2,013


In [14]:
# Deteksi dinamis kolom yang kemungkinan identity fields, berdasarkan nama kolom aktual
# (case-insensitive, tidak hardcode urutan/kapitalisasi dari deskripsi Kaggle)
candidate_keywords = [
    "customer_id", "first_name", "last_name", "name", "email",
    "phone", "address", "dob", "birth", "device_id"
]

actual_cols_lower = {c.lower(): c for c in df_raw.columns}
found_identity_like_cols = {}
for kw in candidate_keywords:
    matches = [orig for lower, orig in actual_cols_lower.items() if kw in lower]
    if matches:
        found_identity_like_cols[kw] = matches

print("Kolom yang namanya mengandung keyword identity (perlu dikonfirmasi manual di Section G):")
for kw, cols in found_identity_like_cols.items():
    print(f"  keyword='{kw}' -> {cols}")


Kolom yang namanya mengandung keyword identity (perlu dikonfirmasi manual di Section G):
  keyword='customer_id' -> ['customer_id']
  keyword='first_name' -> ['first_name']
  keyword='last_name' -> ['last_name']
  keyword='name' -> ['first_name', 'last_name']
  keyword='email' -> ['email']
  keyword='phone' -> ['phone_number']
  keyword='address' -> ['address']
  keyword='dob' -> ['dob']
  keyword='device_id' -> ['device_id(s)']


In [15]:
# Duplicate count per kolom identity-like yang ditemukan (bukan kesimpulan customer duplikat,
# hanya sinyal awal untuk field classification & blocking key candidate di notebook berikutnya)
dup_per_col = []
for kw, cols in found_identity_like_cols.items():
    for col in cols:
        non_null = df_raw[col].dropna()
        dup_count = non_null.duplicated(keep=False).sum()
        dup_per_col.append({
            "column": col,
            "non_null_rows": len(non_null),
            "duplicate_value_rows": dup_count,
            "duplicate_pct_of_non_null": round(dup_count / len(non_null) * 100, 2) if len(non_null) else None
        })

pd.DataFrame(dup_per_col).sort_values("duplicate_pct_of_non_null", ascending=False)


,column,non_null_rows,duplicate_value_rows,duplicate_pct_of_non_null
1,first_name,50000,47293,94.59
3,first_name,50000,47293,94.59
8,dob,50000,46110,92.22
4,last_name,50000,46105,92.21
2,last_name,50000,46105,92.21
6,phone_number,50000,5714,11.43
5,email,48960,4699,9.60
0,customer_id,50000,3534,7.07
7,address,50000,3534,7.07
9,device_id(s),50000,3534,7.07


## E. Cardinality

In [16]:
cardinality_summary = pd.DataFrame({
    "n_unique": df_raw.nunique(dropna=True),
    "n_rows": len(df_raw),
}).assign(
    uniqueness_ratio=lambda d: (d["n_unique"] / d["n_rows"]).round(4)
).sort_values("uniqueness_ratio", ascending=False)

cardinality_summary


,n_unique,n_rows,uniqueness_ratio
customer_id,48200,50000,0.9640
address,48200,50000,0.9640
device_id(s),48200,50000,0.9640
phone_number,46595,50000,0.9319
email,46363,50000,0.9273
city,24534,50000,0.4907
dob,17733,50000,0.3547
last_name,7228,50000,0.1446
first_name,5429,50000,0.1086
signup_date,4383,50000,0.0877


In [17]:
# Frequency distribution untuk kolom dengan cardinality rendah (kandidat categorical field)
low_cardinality_cols = cardinality_summary[cardinality_summary["uniqueness_ratio"] < 0.05].index.tolist()
print(f"Kolom dengan uniqueness ratio < 5% (kandidat categorical): {low_cardinality_cols}")

for col in low_cardinality_cols:
    print(f"\n--- {col} ---")
    print(df_raw[col].value_counts(dropna=False).head(10))


Kolom dengan uniqueness ratio < 5% (kandidat categorical): ['country', 'state', 'gender', 'source']

--- country ---
country
Korea                                                  416
Congo                                                  393
Palau                                                  245
Switzerland                                            239
United States Virgin Islands                           238
Northern Mariana Islands                               238
Mayotte                                                236
British Indian Ocean Territory (Chagos Archipelago)    235
Belize                                                 234
Botswana                                               234
Name: count, dtype: int64

--- state ---
state
Vermont       1051
Kansas        1050
Nebraska      1043
New York      1043
Arkansas      1043
California    1042
Arizona       1039
Oregon        1037
Iowa          1036
Oklahoma      1034
Name: count, dtype: int64

--- gender ---
gender

## E2. customer_id uniqueness - root cause overconfidence

`customer_id` sering diasumsikan sebagai primary key unik. Cek langsung — **bukan** unique.


In [ ]:
cid_counts = df_raw['customer_id'].value_counts(dropna=False)
n_unique_cid = df_raw['customer_id'].nunique(dropna=False)
n_missing_cid = df_raw['customer_id'].isna().sum()
n_blank_cid = (df_raw['customer_id'].astype(str).str.strip() == '').sum() - n_missing_cid
n_dup_ids = (cid_counts > 1).sum()
n_extra_rows = len(df_raw) - n_unique_cid
exact_dup_rows = df_raw.duplicated(keep=False).sum()

print(f"unique     : {n_unique_cid:,} / {len(df_raw):,} (ratio {n_unique_cid/len(df_raw):.4f})")
print(f"missing    : {n_missing_cid:,}")
print(f"blank      : {n_blank_cid:,}")
print(f"IDs appearing >1       : {n_dup_ids:,}")
print(f"  2x : {(cid_counts==2).sum():,}, 3x : {(cid_counts==3).sum():,}, 4x : {(cid_counts==4).sum():,}")
print(f"Extra dup rows         : {n_extra_rows:,}")
print(f"Exact dup rows (all cols identical): {exact_dup_rows:,}")

if n_unique_cid == len(df_raw):
    print("\n-> customer_id UNIK — bisa jadi blocking key.")
else:
    print("\n-> kolom TIDAK UNIK - bukan primary key. Jangan pakai sebagai blocking/matching feature.")
    print("   Tidak ada kolom dengan uniqueness_ratio == 1.0 -> tidak ada natural unique key di dataset ini.")


unique     : 48,200 / 50,000 (ratio 0.9640)
missing    : 0
blank      : 0
IDs appearing >1       : 1,734
  2x : 1,669, 3x : 64, 4x : 1
Extra dup rows         : 1,800
Exact dup rows (all cols identical): 2,013

-> customer_id TIDAK UNIK — bukan primary key. Jangan pakai sebagai blocking/matching feature.
   Tidak ada kolom dengan uniqueness_ratio == 1.0 -> tidak ada natural unique key di dataset ini.


In [20]:
# Apakah same-customer_id = orang yang sama? Cek core fields (email/phone/dob)
dup_cids = cid_counts[cid_counts > 1].index
dup_rows = df_raw[df_raw['customer_id'].isin(dup_cids)]
exact_mask = df_raw.duplicated(keep=False)
n_exact_in_dup = dup_rows[exact_mask[dup_rows.index].values].shape[0]
n_diff_in_dup = dup_rows[~exact_mask[dup_rows.index].values].shape[0]
print(f"Same-customer_id rows: {len(dup_rows):,} (exact dup {n_exact_in_dup:,}, diff {n_diff_in_dup:,})")

core_same = core_diff = 0
for cid in dup_cids:
    r = df_raw[df_raw['customer_id'] == cid]
    if r['email'].nunique()==1 and r['phone_number'].nunique()==1 and r['dob'].nunique()==1:
        core_same += 1
    else:
        core_diff += 1
print(f"Core identity (email+phone+dob) identical : {core_same:,} ({core_same/len(dup_cids)*100:.1f}%)")
print(f"Core identity differs                    : {core_diff:,} ({core_diff/len(dup_cids)*100:.1f}%)")
print("\nKesimpulan: ~98% same-customer_id punya email/phone/dob identik.")
print("Sisa 39 IDs: email=NaN di kedua baris, phone/dob/address identik -> tetap orang sama.")
print("Artinya silver-standard positives adalah pasangan trivially identical -> penyebab overconfidence Splink.")


Same-customer_id rows: 3,534 (exact dup 2,013, diff 1,521)
Core identity (email+phone+dob) identical : 1,695 (97.8%)
Core identity differs                    : 39 (2.2%)

Kesimpulan: ~98% same-customer_id punya email/phone/dob identik.
Sisa 39 IDs: email=NaN di kedua baris, phone/dob/address identik -> tetap orang sama.
Artinya silver-standard positives adalah pasangan trivially identical -> penyebab overconfidence Splink.


In [21]:
# Verifikasi: tidak ada kolom dengan uniqueness_ratio == 1.0
uniq = df_raw.nunique(dropna=False)
uniq_ratio = (uniq / len(df_raw)).round(4)
fully_unique = uniq_ratio[uniq_ratio == 1.0]
print("Kolom fully unique (ratio==1.0):", fully_unique.index.tolist() if len(fully_unique) else "TIDAK ADA")
print(uniq_ratio.sort_values(ascending=False).to_string())


Kolom fully unique (ratio==1.0): TIDAK ADA
customer_id     0.9640
address         0.9640
device_id(s)    0.9640
phone_number    0.9319
email           0.9273
city            0.4907
dob             0.3547
last_name       0.1446
first_name      0.1086
signup_date     0.0877
country         0.0049
state           0.0010
gender          0.0001
source          0.0001


## F. Data quality

Dicek hanya untuk kolom yang benar-benar ada di dataset (dari Section B), bukan diasumsikan dari deskripsi Kaggle.

In [24]:
text_cols = df_raw.select_dtypes(include="object").columns.tolist()
print(f"Kolom bertipe text/object: {text_cols}")


Kolom bertipe text/object: ['customer_id', 'first_name', 'last_name', 'email', 'phone_number', 'gender', 'dob', 'signup_date', 'address', 'city', 'state', 'country', 'device_id(s)', 'source']


C:\Users\User\AppData\Local\Temp\ipykernel_11536\1625751716.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df_raw.select_dtypes(include="object").columns.tolist()


In [25]:
# Leading/trailing whitespace
whitespace_issue = {}
for col in text_cols:
    s = df_raw[col].dropna().astype(str)
    issue_count = (s != s.str.strip()).sum()
    if issue_count > 0:
        whitespace_issue[col] = issue_count

print("Kolom dengan leading/trailing whitespace:")
print(whitespace_issue if whitespace_issue else "Tidak ditemukan.")


Kolom dengan leading/trailing whitespace:
{'first_name': np.int64(3124), 'last_name': np.int64(3134)}


In [26]:
# Inkonsistensi kapitalisasi: value yang sama secara case-insensitive tapi berbeda case
def capitalization_inconsistency(series):
    s = series.dropna().astype(str)
    lower_groups = s.str.lower().value_counts()
    # ambil grup lower yang punya >1 variasi case asli
    variants = s.groupby(s.str.lower()).nunique()
    inconsistent = variants[variants > 1]
    return len(inconsistent)

cap_issue = {}
for col in text_cols:
    n = df_raw[col].nunique(dropna=True)
    if n == 0 or n > 20000:
        continue  # skip kolom terlalu high-cardinality (mis. UUID/id) biar tidak lambat
    inconsistent_groups = capitalization_inconsistency(df_raw[col])
    if inconsistent_groups > 0:
        cap_issue[col] = inconsistent_groups

print("Kolom dengan inkonsistensi kapitalisasi (jumlah grup nilai yang punya >1 variasi case):")
print(cap_issue if cap_issue else "Tidak ditemukan / tidak dicek (kolom terlalu high-cardinality).")


Kolom dengan inkonsistensi kapitalisasi (jumlah grup nilai yang punya >1 variasi case):
{'first_name': 582, 'last_name': 890}


In [27]:
# Format email: cek kolom yang namanya mengandung 'email'
email_cols = [c for c in df_raw.columns if "email" in c.lower()]
email_pattern = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")

for col in email_cols:
    s = df_raw[col].dropna().astype(str)
    invalid_format = (~s.str.match(email_pattern)).sum()
    has_upper = s.str.contains(r"[A-Z]").sum()
    has_space = s.str.contains(r"\s").sum()
    print(f"Kolom '{col}': total non-null={len(s):,}, format tidak valid={invalid_format:,}, "
          f"mengandung huruf besar={has_upper:,}, mengandung spasi={has_space:,}")


Kolom 'email': total non-null=48,960, format tidak valid=0, mengandung huruf besar=0, mengandung spasi=0


In [28]:
# Format phone: cek kolom yang namanya mengandung 'phone'
phone_cols = [c for c in df_raw.columns if "phone" in c.lower()]

for col in phone_cols:
    s = df_raw[col].dropna().astype(str)
    # pola panjang digit unik untuk melihat variasi format (tanpa mengasumsikan country code)
    digit_lengths = s.str.replace(r"\D", "", regex=True).str.len().value_counts().sort_index()
    has_symbols = s.str.contains(r"[+\-\(\)\s\.]").sum()
    print(f"Kolom '{col}': total non-null={len(s):,}, mengandung simbol format={has_symbols:,}")
    print(f"  Distribusi panjang digit (setelah simbol dibuang):")
    print(digit_lengths)


Kolom 'phone_number': total non-null=50,000, mengandung simbol format=45,940
  Distribusi panjang digit (setelah simbol dibuang):
phone_number
3        36
4      1808
5       212
7         1
8        83
9       744
10    15191
13     8047
14     7841
15     8067
16     3925
17     2016
18     2029
Name: count, dtype: int64


In [29]:
# Format address: cek kolom yang namanya mengandung 'address'
address_cols = [c for c in df_raw.columns if "address" in c.lower()]

for col in address_cols:
    s = df_raw[col].dropna().astype(str)
    has_upper_only = s.str.isupper().sum()
    has_lower_only = s.str.islower().sum()
    avg_len = s.str.len().mean()
    print(f"Kolom '{col}': total non-null={len(s):,}, ALL CAPS={has_upper_only:,}, "
          f"all lowercase={has_lower_only:,}, rata-rata panjang karakter={avg_len:.1f}")


Kolom 'address': total non-null=50,000, ALL CAPS=597, all lowercase=0, rata-rata panjang karakter=22.4


## G. Field classification

Klasifikasi berikut diisi berdasarkan hasil aktual Section B–F terhadap **50.000 baris x 14 kolom** (semua bertipe str). Perlu dicatat: cell load pada file ini masih tanpa `sep=";"` — dataset sebenarnya delimiter `;`. Jalankan ulang dengan `df_raw = pd.read_csv(RAW_PATH, sep=";", encoding="utf-8")` sebelum mensahkan angka di bawah.

| Kategori | Kolom | Alasan (dari data aktual) |
|---|---|---|
| **Identity / Matching Fields (kuat)** | `email`, `phone_number` | Cardinality tinggi: `email` n_unique=46.363 (ratio 0.927), `phone_number` n_unique=46.595 (ratio 0.932). Duplicate rendah-bermakna: 9.6% dan 11.4%. `email`: 0 invalid, 0 uppercase, 0 spasi (48.960 non-null). `phone_number`: 45.940 (91.9%) mengandung simbol format dan 29.966 extension `xNNNNN` — wajib dipisah sebelum standardisasi. Catatan: 1.550 email diawali `shared` — email TIDAK selalu 1:1 orang. |
| **Identity / Matching Fields (lemah, pendukung)** | `first_name`, `last_name`, `dob` | Cardinality rendah: ratio 0.109 / 0.145 / 0.355. Duplicate sangat tinggi 94.6% / 92.2% / 92.2% — ruang nilai terbatas, WAJIB fuzzy comparison. Nama kotor: whitespace leading/trailing (3.124 / 3.134), kapitalisasi tidak konsisten (582 / 890 grup), sisipan karakter (`Tho8mas`, `Tara*`, `Dav1id`, `Brittany@`). |
| **Identity / Matching Fields (struktural)** | `address`, `device_id(s)` | Ratio 0.964 (sama dengan `customer_id`), duplicate 7.07%, berduplikasi bersamaan — 1:1 dengan profile. Sinyal pendukung, bukan blocking key utama. `device_id(s)`: 0 nilai mengandung `;` — tidak perlu split semicolon. |
| **Supporting Fields** | `gender`, `country`, `state`, `city` | `gender` 3 unique, `country` 243, `state` 50, `city` 24.534. Tidak cukup diskriminatif; berguna untuk validasi silang. `country` 243 unique — variasi penulisan, perlu standardisasi. |
| **Metadata / Administrative Fields** | `signup_date`, `source` | Bukan identitas. `signup_date` n_unique 4.383; `source` 3 nilai (referral/app/web) + 1.251 missing. |
| **Source Record Identifier (bukan matching feature)** | `customer_id` | n_unique 48.200 (ratio 0.964), 3.534 duplicate rows (7.07%). Hanya untuk reference pair generation, diagnostic, evaluation — BUKAN comparison/blocking/similarity feature. |


## H. Output — Ringkasan Notebook 01 (berdasarkan angka aktual)

```text
Dataset size              : 50.000 baris x 14 kolom (dengan sep=";")

Columns                   : customer_id, first_name, last_name, email, phone_number,
                             gender, dob, signup_date, address, city, state, country,
                             device_id(s), source

Data types                : SEMUA kolom bertipe str; dob & signup_date belum ter-parse
                             sebagai datetime (format aktual dd/mm/yyyy)

Missing values            : source          1.251 (2.50%)  <- paling parah
                             email           1.040 (2.08%)
                             kolom lain          0 (0.00%)

Exact duplicates          : 2.013 baris (duplicated(keep=False))

Unique statistics         : HIGH (>0.90): customer_id/address/device_id(s) 0.964,
                             phone_number 0.932, email 0.927
                             MEDIUM: city 0.491, dob 0.355
                             LOW: last_name 0.145, first_name 0.109, signup_date 0.088,
                             country 0.005, state 0.001, gender & source 0.0001

Duplicate per kolom       : first_name 94.59% | dob 92.22% | last_name 92.21%
                             city 69.23% | phone_number 11.43% | email 9.60%
                             customer_id/address/device_id(s) 7.07% (3.534 baris)

Potential identity fields : kuat -> email, phone_number
                             lemah/pendukung -> first_name, last_name, dob
                             struktural -> address, device_id(s)

Potential supporting fields: gender, country, state, city

Metadata fields            : signup_date, source
customer_id                : source record identifier (dataset-provided reference identity,
                             bukan feature matching)

Initial data-quality findings:
  - Whitespace (leading/trailing): first_name 3.124 baris, last_name 3.134 baris
  - Kapitalisasi tidak konsisten: first_name 582 grup, last_name 890 grup nilai
  - Sisipan karakter acak di nama: 'Tho8mas', 'Tara*', 'Dav1id', 'Brittany@', 'Ha0rt'
  - Email: format bersih (0 invalid/uppercase/spasi), TAPI 1.550 email prefix 'shared'
  - Phone: 45.940 (91.9%) simbol format; 29.966 extension '...xNNNNN' -> pisah jadi
    phone_std + phone_extension
  - Address: 597 baris ALL CAPS, rata-rata panjang 22,4 karakter
  - country: 243 unique value -> variasi penulisan (butuh standardisasi)
```

### Open items

1. **Load cell file ini masih tanpa `sep=";"`** — dataset delimiter `;`. Perbaiki
   sebelum menjalankan ulang angka di atas.
2. **`device_id(s)` tanpa semicolon** — kolom tunggal, tidak perlu split.
3. **Email & source missing** adalah NaN aktual (bukan blank string).

**Next:** `02_standardization.ipynb`.


Hanya Test

In [ ]:
# 1) Contoh exact duplicate rows
df_raw[df_raw.duplicated(keep=False)].sort_values('customer_id').head(20)

,customer_id,first_name,last_name,email,phone_number,gender,dob,signup_date,address,city,state,country,device_id(s),source
1393,005a7521-db6a-406e-b14e-97bf259f96fb,Alexandra,Ellison,alexandra.ellison003@yahoo.com,(543)768-3047x8198,M,09/06/2005,29/04/2018,427 Walton Canyon,Jamiechester,Arkansas,France,14edf41b-5067-45da-834f-7318116cc466,referral
5785,005a7521-db6a-406e-b14e-97bf259f96fb,Alexandra,Ellison,alexandra.ellison003@yahoo.com,(543)768-3047x8198,M,09/06/2005,29/04/2018,427 Walton Canyon,Jamiechester,Arkansas,France,14edf41b-5067-45da-834f-7318116cc466,referral
400,00b15609-c67f-4057-a23e-846120a5a078,John,Paul,john.paul309@yahoo.com,+1-045-245-5744x387,M,19/01/1960,22/03/2022,95225 Davis Passage,Orozcochester,South Carolina,Syrian Arab Republic,1f8da822-3fad-40c4-9c0a-2b9a8adb3325,referral
43335,00b15609-c67f-4057-a23e-846120a5a078,John,Paul,john.paul309@yahoo.com,+1-045-245-5744x387,M,19/01/1960,22/03/2022,95225 Davis Passage,Orozcochester,South Carolina,Syrian Arab Republic,1f8da822-3fad-40c4-9c0a-2b9a8adb3325,referral
17609,00d5aa6f-3b76-40da-90a7-1526ae3211b3,Kelly,Bennett,shared1450@gmail.com,605.131.6162x349,F,01/09/1976,05/12/2021,7015 James Falls,Kristenport,Arizona,Oman,6b05d63b-4d11-4c6d-9e14-770f657c6a39,app
43836,00d5aa6f-3b76-40da-90a7-1526ae3211b3,Kelly,Bennett,shared1450@gmail.com,605.131.6162x349,F,01/09/1976,05/12/2021,7015 James Falls,Kristenport,Arizona,Oman,6b05d63b-4d11-4c6d-9e14-770f657c6a39,app
27020,00d9d551-d21c-4355-a3a6-6a76180194c8,Brad,Jones,brad.jones644@gmail.com,+1-710-203-1409x51948,F,19/04/1960,16/04/2020,302 Taylor Groves Suite 333,South Stephanie,South Carolina,New Zealand,c133b493-008d-4ec2-a934-b13782fbb1e2,app
36079,00d9d551-d21c-4355-a3a6-6a76180194c8,Brad,Jones,brad.jones644@gmail.com,+1-710-203-1409x51948,F,19/04/1960,16/04/2020,302 Taylor Groves Suite 333,South Stephanie,South Carolina,New Zealand,c133b493-008d-4ec2-a934-b13782fbb1e2,app
41230,011ecd43-e9f7-429c-b873-8a4f1ae4b96b,Mary,Rhodes,mary.rhodes599@yahoo.com,217-294-1398,F,04/07/2002,06/05/2018,2479 Robert Drive Suite 425,New Rodneyshire,Missouri,Nicaragua,3aab6405-b9dd-4e9d-b5f7-b50bda04d256,referral
47769,011ecd43-e9f7-429c-b873-8a4f1ae4b96b,Mary,Rhodes,mary.rhodes599@yahoo.com,217-294-1398,F,04/07/2002,06/05/2018,2479 Robert Drive Suite 425,New Rodneyshire,Missouri,Nicaragua,3aab6405-b9dd-4e9d-b5f7-b50bda04d256,referral


In [ ]:
# 2) Baris dengan customer_id duplikat TAPI BUKAN exact duplicate row
# -> ini untuk konfirmasi temuan #1 di atas (dirty duplicate profile vs ID korup)
dup_cid_mask = df_raw.duplicated(subset=['customer_id'], keep=False)
exact_mask = df_raw.duplicated(keep=False)
partial_dup_cid = df_raw[dup_cid_mask & ~exact_mask]
print(f"Jumlah baris: {len(partial_dup_cid)}")
partial_dup_cid.sort_values('customer_id').head(20)

Jumlah baris: 1521


,customer_id,first_name,last_name,email,phone_number,gender,dob,signup_date,address,city,state,country,device_id(s),source
30391,00682ed4-54ac-434e-b800-a4dfa3775f10,Melissa,Peck,shared1349@hotmail.com,6610153591,M,19/01/2001,20/11/2024,893 Chavez Landing Suite 262,New Carrieville,Hawaii,Serbia,002db8ff-ed9f-48ed-ac1b-9f977421bc66,app
38030,00682ed4-54ac-434e-b800-a4dfa3775f10,Melissa,PECK,shared1349@hotmail.com,6610153591,M,19/01/2001,20/11/2024,893 Chavez Landing Suite 262,New Carrieville,Hawaii,Serbia,002db8ff-ed9f-48ed-ac1b-9f977421bc66,app
17061,0070c246-7754-46db-a55f-b14188840068,JOSEPH,sMITH,joseph.smith734@hotmail.com,001-740-556-0991,F,10/05/1996,15/05/2021,2067 Allen Ways,New Patricia,Louisiana,Palau,c33a812d-fb6d-485e-be77-fc0e1526afeb,web
39917,0070c246-7754-46db-a55f-b14188840068,Joseph,Smith,joseph.smith734@hotmail.com,001-740-556-0991,F,10/05/1996,15/05/2021,2067 Allen Ways,New Patricia,Louisiana,Palau,c33a812d-fb6d-485e-be77-fc0e1526afeb,web
34468,00c929d1-753c-49a8-af28-eb3ffe0ad087,tINA,Frost,caitlin.richardson319@hotmail.com,(142)446-4128,M,01/05/1996,01/11/2016,531 Michael Fords Suite 310,Lake Michaelberg,Wyoming,Gabon,14c1d235-28f3-4376-ba5c-758f17d67c68,referral
43939,00c929d1-753c-49a8-af28-eb3ffe0ad087,Tina,Frost,caitlin.richardson319@hotmail.com,(142)446-4128,M,01/05/1996,01/11/2016,531 Michael Fords Suite 310,Lake Michaelberg,Wyoming,Gabon,14c1d235-28f3-4376-ba5c-758f17d67c68,referral
28123,0167fd2f-c24d-4f99-a133-6678590ae380,Michael,Valdez,michael.valdez657@gmail.com,989040312,F,26/02/1963,06/12/2018,2137 Myers Cliffs,Bakerland,Oregon,Angola,530107b6-b7f9-4da1-a3c8-bb4dcf9e5cb5,app
32696,0167fd2f-c24d-4f99-a133-6678590ae380,Mic2hael,VALDEZ,michael.valdez657@gmail.com,989040312,F,26/02/1963,06/12/2018,2137 Myers Cliffs,Bakerland,Oregon,Angola,530107b6-b7f9-4da1-a3c8-bb4dcf9e5cb5,app
25598,01695a54-eb34-4506-873d-669cc875b8d0,Sean,Cook2e,sean.cooke027@yahoo.com,-4151,F,03/12/1959,26/10/2021,8290 Cynthia Fords Apt. 413,North Brenthaven,California,Lao People's Democratic Republic,3bbbfba2-0f31-40bc-b543-f2eabff95598,referral
26181,01695a54-eb34-4506-873d-669cc875b8d0,Sean,Cooke,sean.cooke027@yahoo.com,-4151,F,03/12/1959,26/10/2021,8290 Cynthia Fords Apt. 413,North Brenthaven,California,Lao People's Democratic Republic,3bbbfba2-0f31-40bc-b543-f2eabff95598,referral


In [ ]:
# 3) Konfirmasi missing first_name yang sebenarnya (blank string, bukan NaN)
df_raw[df_raw['first_name'].astype(str).str.strip() == '']

,customer_id,first_name,last_name,email,phone_number,gender,dob,signup_date,address,city,state,country,device_id(s),source


In [ ]:
# 4) Sample nomor telepon dengan digit sangat panjang (15-18 digit) untuk lihat pola aslinya
df_raw.loc[df_raw['phone_number'].astype(str).str.replace(r'\D', '', regex=True).str.len() >= 15, 'phone_number'].head(20)

1         501.347.4824x64633
2      +1-268-822-6332x26814
6      001-063-045-0948x4636
14       (589)341-8236x24713
15      +1-712-046-0252x3070
20     +1-362-979-6775x98610
21      +1-959-000-2424x3019
22     +1-631-072-5576x19258
25      +1-822-082-7385x4001
34        432.270.3188x08867
35      +1-100-400-1071x4180
42      001-986-706-3302x097
45       (536)861-2563x78937
48        570-173-8500x25110
49      +1-690-938-2229x0926
50    001-085-209-1544x10230
54    001-779-177-7444x66359
56      +1-073-626-9897x9185
58     +1-813-237-9098x93797
61      001-787-063-7028x595
Name: phone_number, dtype: str